# 端到端动态Alpha模型 — 复现报告

**论文**：招商证券《端到端的动态Alpha模型》(2023)

**核心思路**：用 MLP 神经网络替代传统多因子选股中的人工加权合成，直接从原始因子端到端学习收益预测。

```
输入(11因子) → MLP([11→64→64→1]) → 隐因子层(正交惩罚) → 预测收益截面排名 → IC Loss
```

**实验设置**：50只A股 | 2018-2023 | 11因子 | 6窗口滚动训练 | 5轮17实验超参数调优

**环境**：Python 3.12.7 + PyTorch(CUDA cu124) + RTX 4060 8GB

In [ ]:
# ===== 环境设置 =====
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown, HTML

# 将项目根目录加入路径
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    FACTOR_COLS, FACTOR_DIRECTION, STOCK_POOL, DEVICE,
    CHECKPOINT_DIR, LOG_DIR, PROCESSED_DATA_DIR,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

print(f"PyTorch {torch.__version__}  |  Device: {DEVICE}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 1. 股票池与因子体系

In [ ]:
# 行业分布
industry_map = {
    "银行": ["600036", "601398", "601288", "601166", "600000", "601328"],
    "保险/券商": ["601318", "600030", "601601", "601688"],
    "食品饮料": ["600519", "000858", "000568", "000333", "600887"],
    "科技/电子": ["002415", "300750", "002475", "002230", "600588", "002049"],
    "医药": ["600276", "000538", "300760", "600196", "002007"],
    "地产/建材": ["000002", "600048", "600585"],
    "钢铁/有色": ["600019", "601899", "600362"],
    "化工": ["600309", "600426", "002493"],
    "电力/公用": ["600900", "601985", "600025"],
    "建筑": ["601668", "601186"],
    "通信/传媒": ["600050", "000063"],
    "石油/煤炭": ["601857", "601088", "601225"],
    "新能源": ["601012", "002129"],
    "汽车": ["600104", "002594"],
    "家电": ["000651"],
}

industry_counts = {k: len(v) for k, v in industry_map.items()}
df_ind = pd.DataFrame(industry_counts.items(), columns=["行业", "股票数"]).sort_values("股票数", ascending=False)
display(df_ind.style.bar(subset=["股票数"], color="steelblue"))
print(f"总计: {sum(industry_counts.values())} 只股票, {len(industry_map)} 个行业")

In [ ]:
# 因子体系
factor_df = pd.DataFrame([
    {"类别": "估值", "因子": "ep", "名称": "盈利收益率 (E/P)", "方向": "+"},
    {"类别": "估值", "因子": "bp", "名称": "账面市值比 (B/P)", "方向": "+"},
    {"类别": "成长", "因子": "roe_growth", "名称": "ROE 增长率", "方向": "+"},
    {"类别": "成长", "因子": "profit_growth", "名称": "净利润增长率", "方向": "+"},
    {"类别": "成长", "因子": "revenue_growth", "名称": "营收增长率", "方向": "+"},
    {"类别": "经营", "因子": "roe", "名称": "净资产收益率", "方向": "+"},
    {"类别": "经营", "因子": "asset_turnover", "名称": "总资产周转率", "方向": "+"},
    {"类别": "流动性", "因子": "turnover_rate", "名称": "换手率", "方向": "-"},
    {"类别": "流动性", "因子": "amplitude", "名称": "振幅", "方向": "-"},
    {"类别": "技术", "因子": "momentum_20", "名称": "20日动量", "方向": "+"},
    {"类别": "技术", "因子": "reversal_5", "名称": "5日反转", "方向": "-"},
])

display(HTML(factor_df.to_html(index=False, classes="table table-striped")))
print(f"共 {len(factor_df)} 个因子，覆盖 {factor_df['类别'].nunique()} 个类别")

---
## 2. 数据管线

In [ ]:
# 加载预处理数据
processed_path = PROCESSED_DATA_DIR / "processed_data.csv"
df = pd.read_csv(processed_path, parse_dates=["date"])

print(f"数据规模: {len(df):,} 行 × {len(df.columns)} 列")
print(f"日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"股票数量: {df['stock_code'].nunique()}")
print(f"交易日数: {df['date'].nunique()}")
print(f"\n因子列: {list(FACTOR_COLS)}")
print(f"\n标签统计:")
display(df["label"].describe().to_frame().T)

In [ ]:
# 验证截面标准化：随机抽样5个截面，检查均值和标准差
sample_dates = np.random.choice(df["date"].unique(), min(5, df["date"].nunique()), replace=False)
sample_dates = sorted(sample_dates)

results = []
for d in sample_dates:
    sub = df[df["date"] == d]
    means = sub[FACTOR_COLS].mean()
    stds = sub[FACTOR_COLS].std()
    results.append({
        "date": str(d.date()),
        "n_stocks": len(sub),
        "mean_range": f"[{means.min():.4f}, {means.max():.4f}]",
        "std_range": f"[{stds.min():.4f}, {stds.max():.4f}]",
    })

valid_df = pd.DataFrame(results)
display(valid_df)

# 整体检查
all_means = df.groupby("date")[FACTOR_COLS].mean()
all_stds = df.groupby("date")[FACTOR_COLS].std()
print(f"\n全局均值范围: [{all_means.mean().mean():.4f}]  (期望 ≈ 0)")
print(f"全局标准差均值: [{all_stds.mean().mean():.4f}]  (期望 ≈ 1)")

# 可视化因子分布
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, col in enumerate(FACTOR_COLS):
    ax = axes[i // 4][i % 4]
    ax.hist(df[col].dropna(), bins=50, density=True, alpha=0.7, color="steelblue")
    ax.set_title(col)
    ax.axvline(x=0, color="red", linestyle="--", alpha=0.5)
for j in range(len(FACTOR_COLS), 12):
    axes[j // 4][j % 4].set_visible(False)
fig.suptitle("因子分布 (标准化后)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 3. 模型架构

In [ ]:
from models.mlp_alpha import MLPAlphaModel
from models.linear_alpha import LinearAlphaModel

n_factors = len(FACTOR_COLS)

# MLP 模型
mlp = MLPAlphaModel(n_factors, hidden_dims=(64, 64), activation="leaky_relu")
mlp_params = sum(p.numel() for p in mlp.parameters())
mlp_trainable = sum(p.numel() for p in mlp.parameters() if p.requires_grad)

# Linear 模型
linear = LinearAlphaModel(n_factors)
linear_params = sum(p.numel() for p in linear.parameters())

print(f"=== MLP 模型 ===")
print(f"结构: [11 -> 64 -> 64 -> 1]")
print(f"参数量: {mlp_params:,} (可训练: {mlp_trainable:,})")
print(f"\n{mlp}")
print(f"\n=== Linear 模型 ===")
print(f"结构: [11 -> 1]")
print(f"参数量: {linear_params:,}")
print(f"\n{linear}")

### 关键设计决策

| 设计 | 理由 |
|------|------|
| **末层无激活函数** | 标签是截面zscore（有正有负），ReLU会强制>=0导致IC为负 |
| **BatchNorm (非LayerNorm)** | 小batch下LayerNorm灾难（-4.07% IC） |
| **LeakyReLU** | 避免Sigmoid梯度消失，调优最优激活函数(+0.94%) |
| **正交惩罚 λ=0.1** | 强正交适合小样本，避免隐因子共线性 |
| **IC-based早停** | 量化场景关注排序能力，IC比MSE更重要 |
| **无Dropout/WD** | 50只小样本下正则化反而不利（欠拟合非过拟合） |

---
## 4. P1 — 基准模型对比

In [ ]:
# 加载P1基准测试结果（训练历史JSON）
def load_history(model_type, loss_name):
    path = LOG_DIR / f"{model_type}_{loss_name}_history.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

# P1 四种方案对比
configs = [
    ("Linear", "MSE", "linear_mse"),
    ("MLP", "MSE", "mlp_mse"),
    ("MLP", "IC", "mlp_ic"),
    ("MLP", "CCC", "mlp_ccc"),
]

# P1 测试集评估结果（来自REPORT）
p1_results = pd.DataFrame([
    {"Model": "Linear+MSE", "Rank IC": 0.158, "ICIR": 0.81, "IC Win Rate": 0.664},
    {"Model": "MLP+IC", "Rank IC": 0.103, "ICIR": 0.60, "IC Win Rate": 0.560},
    {"Model": "MLP+MSE", "Rank IC": 0.074, "ICIR": 0.43, "IC Win Rate": 0.536},
    {"Model": "MLP+CCC", "Rank IC": 0.021, "ICIR": 0.11, "IC Win Rate": 0.456},
])

display(HTML("<h4>P1 测试集评估结果</h4>"))
display(p1_results.style
    .bar(subset=["Rank IC"], color="steelblue")
    .bar(subset=["ICIR"], color="darkorange")
    .format({"Rank IC": "{:.1%}", "ICIR": "{:.2f}", "IC Win Rate": "{:.1%}"})
)

print("\n关键发现：")
print("- Linear+MSE 表现最佳(IC=15.8%)：50只小样本下简单模型占优")
print("- MLP+IC 第二(10.3%)：IC loss直接优化排序比MSE更合理")
print("- MLP+CCC 最弱(2.1%)：CCC需要更大样本估计分布参数")

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (model_type, loss_name, fname) in enumerate(configs):
    ax = axes[idx // 2][idx % 2]
    history = load_history(model_type.lower(), loss_name.lower())
    if history:
        epochs = range(1, len(history["train_loss"]) + 1)
        color1 = "steelblue"
        color2 = "darkorange"
        
        ax_loss = ax
        ax_loss.plot(epochs, history["train_loss"], label="Train Loss", color=color1, alpha=0.7, linewidth=1)
        ax_loss.plot(epochs, history["val_loss"], label="Val Loss", color=color2, alpha=0.7, linewidth=1)
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_title(f"{model_type} + {loss_name}")
        ax_loss.legend(fontsize=9)
        ax_loss.grid(True, alpha=0.3)
        
        # 标注最佳IC
        best_ic = max(history["val_ic"]) if history["val_ic"] else 0
        ax_loss.text(0.98, 0.98, f"Best Val IC: {best_ic:.3f}",
                    transform=ax_loss.transAxes, ha="right", va="top",
                    fontsize=10, fontweight="bold",
                    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

fig.suptitle("P1 Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 展示已保存的分组收益图
from IPython.display import Image as IPImage

print("=== 分组回测图 (P1) ===")
print("按预测排名分10组，预期：高组收益 > 低组收益，多空对冲为正")

image_files = [
    LOG_DIR / "Linear_MSE_group_return.png",
    LOG_DIR / "MLP_IC_group_return.png",
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, img_path in enumerate(image_files):
    if img_path.exists():
        img = plt.imread(img_path)
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(img_path.stem, fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. P2 — 滚动训练

In [ ]:
# 加载滚动训练对比表
rolling_path = LOG_DIR / "rolling_comparison.csv"
rolling_df = pd.read_csv(rolling_path)

# 按窗口和模型透视
pivot_ic = rolling_df.pivot(index="window", columns="model", values="rank_ic_mean")
pivot_icir = rolling_df.pivot(index="window", columns="model", values="icir")

print("=== 滚动训练 Rank IC 对比 (6窗口) ===")
window_periods = ["2021H1", "2021H2", "2022H1", "2022H2", "2023H1", "2023H2"]
pivot_ic.index = window_periods
display(pivot_ic.style.format("{:.2%}").background_gradient(cmap="RdYlGn", axis=None))

# 平均值
avg_ic = pivot_ic.mean()
print("\n=== 各模型滚动平均 IC ===")
for model, ic in avg_ic.sort_values(ascending=False).items():
    print(f"  {model:15s}: {ic:+.2%}")

print(f"\n关键发现：MLP+IC 平均最优(+{avg_ic.max():.2%})，但跨窗口方差大")
print("2022年系统性低谷(W2/W3)：全模型IC为负，反映市场环境切换")

In [ ]:
# 滚动对比图
rolling_chart_path = LOG_DIR / "rolling_comparison.png"
ic_trend_path = LOG_DIR / "rolling_ic_trend.png"

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

for i, path in enumerate([rolling_chart_path, ic_trend_path]):
    if path.exists():
        img = plt.imread(path)
        axes[i].imshow(img)
        axes[i].axis("off")

plt.tight_layout()
plt.show()

---
## 6. 超参数调优 (5轮17实验)

In [ ]:
# 加载所有调优实验结果
tuning_dir = LOG_DIR / "tuning"
tuning_results = []

for json_path in sorted(tuning_dir.glob("*.json")):
    with open(json_path) as f:
        data = json.load(f)
    summary = data.get("summary", {})
    kwargs = data.get("kwargs", {})
    tuning_results.append({
        "name": data["name"],
        "round": data.get("round", 0),
        "desc": data.get("description", ""),
        "mean_ic": summary.get("mean_ic", 0),
        "std_ic": summary.get("std_ic", 0),
        "min_ic": summary.get("min_ic", 0),
        "max_ic": summary.get("max_ic", 0),
        "positive_windows": summary.get("positive_windows", "0/6"),
        "lr": kwargs.get("lr", 0.0005),
        "lambda_orth": kwargs.get("lambda_orth", 0.01),
        "dropout": kwargs.get("dropout", 0),
        "wd": kwargs.get("weight_decay", 0),
        "activation": kwargs.get("activation", "sigmoid"),
        "hidden_dims": kwargs.get("hidden_dims", "(64, 64)"),
        "use_ln": kwargs.get("use_layer_norm", False),
    })

tuning_df = pd.DataFrame(tuning_results).sort_values("mean_ic", ascending=False)
print(f"加载 {len(tuning_df)} 组调优实验")

# 全排名表
display_cols = ["name", "round", "mean_ic", "positive_windows", "lr", "lambda_orth"]
display(HTML("<h4>全排名 (按Mean IC降序)</h4>"))
display(tuning_df[display_cols].style
    .format({"mean_ic": "{:+.2%}", "lr": "{:.4f}", "lambda_orth": "{:.3f}"})
    .bar(subset=["mean_ic"], color="steelblue")
    .background_gradient(subset=["mean_ic"], cmap="RdYlGn")
)

print(f"\nTop 3: {', '.join(tuning_df['name'].iloc[:3].tolist())}")
print(f"Bottom 3: {', '.join(tuning_df['name'].iloc[-3:].tolist())}")

In [ ]:
# 按轮次可视化
round_names = {1: "R1 正则化", 2: "R2 学习率", 3: "R3 网络架构", 4: "R4 正交λ", 5: "R5 激活函数"}
rounds = sorted(tuning_df["round"].unique())

fig, axes = plt.subplots(1, len(rounds), figsize=(18, 4.5), sharey=True)

for i, r in enumerate(rounds):
    ax = axes[i]
    r_df = tuning_df[tuning_df["round"] == r].sort_values("mean_ic", ascending=True)
    
    colors = ["green" if v > 0 else "red" for v in r_df["mean_ic"]]
    bars = ax.barh(range(len(r_df)), r_df["mean_ic"], color=colors, edgecolor="black", linewidth=0.5)
    
    ax.set_yticks(range(len(r_df)))
    ax.set_yticklabels(r_df["name"], fontsize=8)
    ax.set_title(round_names.get(r, f"Round {r}"), fontsize=11)
    ax.axvline(x=0, color="black", linewidth=0.8)
    ax.grid(True, alpha=0.3, axis="x")
    
    for j, (_, row) in enumerate(r_df.iterrows()):
        ax.text(row["mean_ic"] + (0.003 if row["mean_ic"] >= 0 else -0.02),
                j, f"{row['mean_ic']:+.2%}", va="center", fontsize=7)

fig.suptitle("5轮调优实验结果汇总", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 关键超参数敏感度分析
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 学习率 → IC
r2_data = tuning_df[tuning_df["round"] == 2].sort_values("lr")
axes[0].plot(r2_data["lr"], r2_data["mean_ic"], "o-", color="steelblue", markersize=10, linewidth=2)
axes[0].set_xlabel("Learning Rate")
axes[0].set_ylabel("Mean IC")
axes[0].set_title("LR Sensitivity (R2)")
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color="red", linestyle="--", alpha=0.5)
for _, row in r2_data.iterrows():
    axes[0].annotate(f"{row['mean_ic']:+.2%}", (row["lr"], row["mean_ic"]),
                    textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8)

# 正交λ → IC
r4_data = tuning_df[tuning_df["round"] == 4].sort_values("lambda_orth")
axes[1].plot([0, 0.001, 0.01, 0.1], r4_data["mean_ic"], "s-", color="darkorange", markersize=10, linewidth=2)
axes[1].set_xlabel("Lambda (Orthogonal)")
axes[1].set_ylabel("Mean IC")
axes[1].set_title("Orth Lambda Sensitivity (R4)")
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color="red", linestyle="--", alpha=0.5)
axes[1].set_xscale("symlog", linthresh=0.0005)
for _, row in r4_data.iterrows():
    axes[1].annotate(f"{row['mean_ic']:+.2%}", (row["lambda_orth"], row["mean_ic"]),
                    textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8)

# 网络架构 → IC
r3_data = tuning_df[tuning_df["round"] == 3].copy()
r3_labels = ["[64,64]\n(Baseline)", "[32,32]", "[128,128]", "[64,32,16]"]
colors3 = ["green" if v > 0 else "red" for v in r3_data["mean_ic"]]
axes[2].bar(r3_labels, r3_data["mean_ic"], color=colors3, edgecolor="black", linewidth=0.5)
axes[2].set_ylabel("Mean IC")
axes[2].set_title("Architecture Sensitivity (R3)")
axes[2].grid(True, alpha=0.3, axis="y")
axes[2].axhline(y=0, color="black", linewidth=0.8)
for j, (_, row) in enumerate(r3_data.iterrows()):
    axes[2].text(j, row["mean_ic"] + (0.002 if row["mean_ic"] >= 0 else -0.01),
                f"{row['mean_ic']:+.2%}", ha="center", fontsize=9, fontweight="bold")

fig.suptitle("关键超参数敏感度分析", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n=== 关键洞察 ===")
print("1. 学习率(极差5.21%)：最重要超参数，LR=0.003最优")
print("2. 正交λ：单调递增，λ=0.1最优(+1.81%) — 强正交适合小样本")
print("3. 网络架构：[64,64]是甜点 — 缩小欠拟合，放大过拟合")
print("4. 正则化(WD/Dropout)：对小样本有害，核心问题是欠拟合而非过拟合")
print("5. 激活函数：LeakyReLU(+0.94%) > GELU(+0.81%, 5/6正) > LayerNorm灾难(-4.07%)")

---
## 7. 最优配置与最终结论

In [ ]:
optimal = """
| 参数 | 最优值 | 来源 |
|------|--------|------|
| 模型 | MLP + IC Loss | P1 |
| 架构 | [11 → 64 → 64 → 1] | R3 |
| 归一化 | BatchNorm1d | R5 |
| 激活函数 | LeakyReLU(0.01) | R5 |
| 学习率 | 0.003 (Adam) | R2 |
| 正交 λ | 0.1 | R4 |
| Dropout | 0 (不使用) | R1 |
| Weight Decay | 0 (不使用) | R1 |
| 早停 | patience=50 (验证集IC) | P0 |
| **Mean IC** | **+1.81%** | R4.3 |
"""

display(Markdown(optimal))

# 全排名 Top 6
print("=== 全排名 Top 6 ===")
top6 = tuning_df.head(6)
for i, (_, row) in enumerate(top6.iterrows()):
    medal = ["🥇", "🥈", "🥉"][i] if i < 3 else f"  {i+1}."
    print(f"{medal} {row['name']:20s}  IC={row['mean_ic']:+.2%}  ({row['positive_windows']} 正窗口)")

print(f"\n最优配置对应命令：")
print(f"python train.py --mode tuning --lr 0.003 --lambda_orth 0.1 --activation leaky_relu")

In [ ]:
# 最终汇总：调优历程全景图
all_rounds = tuning_df.groupby("round").agg(
    best_ic=("mean_ic", "max"),
    best_name=("mean_ic", lambda x: tuning_df.loc[x.idxmax(), "name"]),
    n_experiments=("name", "count"),
).reset_index()
all_rounds["round_label"] = all_rounds["round"].map(round_names)

fig, ax = plt.subplots(figsize=(10, 5))

# 累积最优IC
cumulative_best = []
running_max = -float("inf")
for _, row in all_rounds.iterrows():
    running_max = max(running_max, row["best_ic"])
    cumulative_best.append(running_max)

x = range(len(all_rounds))
ax.plot(x, all_rounds["best_ic"], "o-", color="steelblue", linewidth=2, markersize=12, label="Round Best")
ax.plot(x, cumulative_best, "s--", color="darkorange", linewidth=2, markersize=10, label="Cumulative Best")

ax.set_xticks(x)
ax.set_xticklabels(all_rounds["round_label"], fontsize=9)
ax.set_ylabel("Mean IC")
ax.set_title("调优历程: 从 Baseline 到最优配置", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color="red", linestyle="--", alpha=0.5, label="IC=0")

for i, (_, row) in enumerate(all_rounds.iterrows()):
    ax.annotate(f"{row['best_name']}\n{row['best_ic']:+.2%}",
               (i, row["best_ic"]),
               textcoords="offset points", xytext=(0, 15),
               ha="center", fontsize=8,
               bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\n从 Baseline(+0.24%) 到 最优(+1.81%)，提升 {0.0181 - 0.0024:+.2%} IC")

---
## 8. 结论与展望

### 完成情况

✅ P1 基准测试：4种方案对比，Linear+MSE 在小样本下最优  
✅ P2 滚动训练：6窗口×4模型=24次训练，MLP+IC 滚动平均最优  
✅ 调优 R1-R5：17实验，找到最优配置 IC +1.81%  
✅ 项目整理：GitHub 发布 (PowerAdapter-Alpha, MIT License)

### 反直觉发现

1. **强正交优于默认** — λ=0.1（远大于论文默认0.01）最优，单调递增
2. **小样本不宜正则化** — Dropout/WD有害，核心问题是欠拟合
3. **学习率是最重要超参数** — 极差5.21%，远超其他超参
4. **BatchNorm > LayerNorm** — 小batch下LN灾难

### 局限性

- 50只股票样本偏小，限制非线性模型发挥
- dp(股息率)因子缺失，估值维度不完整
- 2022年系统性IC为负，跨市场环境泛化不足

### 展望

- 扩大股票池 50→200+，利用VRAM余量（仅用15%）
- 补充dp因子及更多基本面因子
- 引入Transformer/Attention替代MLP
- 加入市值/行业中性化